# 期現套利 HBT：跨日留倉 Runner

這個 Notebook 是可調參數的薄 runner；底層邏輯放在同資料夾與 `future_spot/arbitrage/` 的 Python 模組。預設執行 2026 年 1–7 月單次連續回測：未平倉部位會帶入下一交易日，不在隔日篩選名單的舊合約也會保留，到期日不得殘倉或自動換月。既有 event NPZ 會直接重用。

In [ ]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd().resolve()
TEST_ROOT = CURRENT_DIR if (CURRENT_DIR / 'backtest_config.py').exists() else CURRENT_DIR / 'future_spot' / 'test'
if not TEST_ROOT.exists():
    raise FileNotFoundError('Open this notebook from future_spot/test or the repository root')
PROJECT_ROOT = TEST_ROOT.parent
WORKSPACE_ROOT = PROJECT_ROOT.parent
for path in (TEST_ROOT, PROJECT_ROOT, WORKSPACE_ROOT, PROJECT_ROOT / 'scripts'):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from IPython.display import Image, display
from backtest_config import default_notebook_args
from backtest_pipeline import run_backtest_pipeline
from report_tables import build_report_tables
from report_plots import save_report_plots

## 1. 參數

In [ ]:
args = default_notebook_args(
    start_date='2026-01-01',
    end_date='2026-07-31',
    carry_positions=True,
    total_capital=50_000_000.0,
    futures_margin_rate=0.20,
    spot_equity_rate=0.40,
    leverage=True,  # False：期貨與現股皆按 100% 自有資金
    order_latency_ms=10.0,
    continue_on_error=True,  # 保留已知缺資料日；下方仍會獨立檢查到期日殘倉
    response_latency_ms=10.0,
    feed_latency_offset_ms=10.0,
    post_first_feed_wait='spot',
    post_first_feed_timeout_ms=5000.0,
    rebuild_hbt_results=False,  # manifest 失效時仍會自動重跑 HBT
)
args.output_dir

## 2. 執行完整回測與檢查跨日持倉

In [ ]:
artifacts = run_backtest_pipeline(args)
display(artifacts.frame('summary').head(20))
display(artifacts.frame('entry_exit_index').head(20))

carry_status = artifacts.frame('position_carry_status')
display(carry_status.loc[carry_status['universe_source'].ne('selected')].head(30))
expiry_violations = carry_status.loc[carry_status['status'].eq('expiry_position_remaining')]
if not expiry_violations.empty:
    raise RuntimeError(f'期貨到期日仍有殘倉：{len(expiry_violations)} 筆')
print('跨日留倉稽核通過：未發現到期日殘倉。')

## 3. 產生資金與績效報表

In [ ]:
reports = build_report_tables(artifacts)
display(reports.frame('symbol_profit').head(30))
display(reports.frame('failure_overview'))
display(reports.frame('roi_summary_including_open'))
display(reports.frame('capital_constraint_summary'))
display(reports.frame('daily_capital_constraint').tail(20))
print(f'Report CSV directory: {reports.output_dir}')

## 4. 儲存 PNG 圖表

In [ ]:
figure_paths = save_report_plots(artifacts, reports)
for name, path in figure_paths.items():
    print(f'{name}: {path}')
    display(Image(filename=str(path)))

## 5. 選擇性逐 pair 檢查

In [ ]:
selected_pair = reports.selected_pair
entry_exit = artifacts.frame('entry_exit_all')
latency = artifacts.frame('latency')
if selected_pair is None:
    print('No pair is available for drill-down.')
else:
    if 'pair_name' in entry_exit.columns:
        display(entry_exit.loc[entry_exit['pair_name'].eq(selected_pair)].head(100))
    if 'pair_name' in latency.columns:
        display(latency.loc[latency['pair_name'].eq(selected_pair)].head(120))